<a href="https://colab.research.google.com/github/sankeawthong/Project-1-Lita-Chatbot/blob/main/Lab03_KNN_and_NaiveBayes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 3 — Neighbours, Odds, and a Scaling Experiment


| | |
|---|---|
| **Time budget** | ~2 hours|
| **Learning objectives** | 1. Implement and visualise K-NN classification — 2. Select K honestly with cross-validation —  3. Demonstrate why distance-based models need scaling —  4. Apply Gaussian and Multinomial Naive Bayes and compare algorithms on evidence — |
| **Datasets** | Iris (150 flowers) · Wine (178 wines, wildly mismatched feature scales) · SMS Spam Collection (5,572 real text messages) |
| **Grading** | Exercises 1–6 + AI-usage disclosure · self-checks give instant feedback |

> **Today you get to be right about something before you're told.** In the lecture I claimed K-NN without scaling is *broken*, not merely worse. Part 3 makes you measure the gap yourself. And Part 5 puts both algorithms on a stopwatch, so the comparison table becomes your data.

⚠️ **First:** File → Save a copy in Drive, rename to `Lab3_<your_student_id>`.

---
## Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

from sklearn.datasets import load_iris, load_wine
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

RNG_SEED = 42
np.random.seed(RNG_SEED)
print("Setup complete ✔")

---
## Part 1 · K-NN on Iris: seeing the neighbourhood

Iris: 150 flowers, 4 measurements, 3 species. We'll use **two** features (petal length and width) so we can *draw* what the algorithm is thinking.

In [ ]:
iris = load_iris()
X_iris = iris.data[:, 2:4]          # petal length, petal width
y_iris = iris.target
print("features used:", iris.feature_names[2:4])
print("classes:", list(iris.target_names))

Xtr_i, Xte_i, ytr_i, yte_i = train_test_split(
    X_iris, y_iris, test_size=0.25, stratify=y_iris, random_state=RNG_SEED)

knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5))
knn.fit(Xtr_i, ytr_i)
print(f"\ntest accuracy (K=5): {knn.score(Xte_i, yte_i):.3f}")

### The picture from the lecture — now yours to make

The helper below paints every point of the plane with the class K-NN would predict there. Run it for K = 1, 15 and 99 and watch the boundary go from jagged to sensible to over-smoothed.

In [ ]:
def plot_boundary(k, X, y, ax, weights="uniform"):
    model = make_pipeline(StandardScaler(), KNeighborsClassifier(k, weights=weights)).fit(X, y)
    h = 0.02
    xx, yy = np.meshgrid(np.arange(X[:,0].min()-.5, X[:,0].max()+.5, h),
                         np.arange(X[:,1].min()-.5, X[:,1].max()+.5, h))
    Z = model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, cmap=ListedColormap(["#DCEEED", "#FCEBDC", "#E4E9F2"]))
    ax.scatter(X[:,0], X[:,1], c=y, cmap=ListedColormap(["#15807E", "#E8912D", "#3E5C76"]),
               s=28, edgecolors="white", linewidths=0.6)
    ax.set_title(f"K = {k}", fontsize=14, fontweight="bold")
    ax.set_xlabel("petal length (cm)"); ax.set_ylabel("petal width (cm)")

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))
for ax, k in zip(axes, [1, 15, 99]):
    plot_boundary(k, Xtr_i, ytr_i, ax)
plt.tight_layout(); plt.show()

**Read the pictures.** K=1 carves out little islands around individual points — including any point sitting in the "wrong" place. K=15 is smooth and follows the real structure. K=99 is voting with almost the entire training set, so local detail disappears entirely.

Small K → high variance (overfitting). Large K → high bias (underfitting). Week 1's curve, made visible.

---
## Part 2 · Choosing K honestly

Rule: **never** pick K by trying values on the test set. Cross-validate on the training data, look at mean *and* spread, then commit.

In [ ]:
k_values = range(1, 41)
means, stds = [], []
for k in k_values:
    scores = cross_val_score(make_pipeline(StandardScaler(), KNeighborsClassifier(k)),
                             Xtr_i, ytr_i, cv=5)
    means.append(scores.mean()); stds.append(scores.std())

means, stds = np.array(means), np.array(stds)
best_k = list(k_values)[int(means.argmax())]

plt.figure(figsize=(9, 4.5))
plt.plot(k_values, means, marker="o", ms=4, color="#15807E", label="CV mean accuracy")
plt.fill_between(k_values, means-stds, means+stds, alpha=0.18, color="#2CA6A4", label="± 1 std")
plt.axvline(best_k, ls="--", color="#E8912D", label=f"best K = {best_k}")
plt.xlabel("K"); plt.ylabel("accuracy"); plt.title("Choosing K by cross-validation (training data only)")
plt.legend(); plt.grid(alpha=.25); plt.show()

print(f"best K = {best_k} → CV accuracy {means.max():.3f} ± {stds[means.argmax()]:.3f}")

Notice two things. First, the curve **rises then falls** — the U-shape (upside down, since higher accuracy is better) that the lecture promised. Second, the shaded band: several values of K are within one standard deviation of the winner, meaning the "best" K is not sharply best. Reporting `K=13` as if it were precisely optimal would be over-claiming; the honest statement is "anything in this range performs comparably, and I chose one."

---
## Part 3 · The scaling experiment

Now the claim from the lecture: *K-NN without scaling isn't slightly worse — it's broken.* Let's measure it. The **Wine** dataset is perfect: its 13 features live on wildly different scales.

In [ ]:
wine = load_wine()
Xw, yw = wine.data, wine.target
scales = pd.DataFrame({"feature": wine.feature_names,
                       "min": Xw.min(axis=0).round(2),
                       "max": Xw.max(axis=0).round(2),
                       "range": (Xw.max(axis=0) - Xw.min(axis=0)).round(2)})
print(scales.sort_values("range", ascending=False).head(5).to_string(index=False))
print("\n^ 'proline' spans ~1,400 units while 'nonflavanoid_phenols' spans ~0.5.")
print("  In a raw Euclidean distance, proline IS the distance.")

In [ ]:
Xtr_w, Xte_w, ytr_w, yte_w = train_test_split(
    Xw, yw, test_size=0.25, stratify=yw, random_state=RNG_SEED)

knn_raw    = KNeighborsClassifier(n_neighbors=5).fit(Xtr_w, ytr_w)
knn_scaled = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)).fit(Xtr_w, ytr_w)

acc_raw    = knn_raw.score(Xte_w, yte_w)
acc_scaled = knn_scaled.score(Xte_w, yte_w)

print(f"K-NN, unscaled features : {acc_raw:.3f}")
print(f"K-NN, standardized      : {acc_scaled:.3f}")
print(f"difference              : {acc_scaled - acc_raw:+.3f}  ({(acc_scaled-acc_raw)*100:+.1f} percentage points)")

**That gap is one line of code.** Same algorithm, same data, same K, same split — the only difference is whether the features were allowed to speak at comparable volume.

Note carefully *where* the scaler lives: inside a `Pipeline`. That's Week 2's rule doing its job — when this pipeline is cross-validated, the scaler re-fits on each fold's training portion, never on validation data.

---
## Part 4 · Naive Bayes, twice

### 4a · GaussianNB on measurements
Same Wine data, a completely different philosophy: no distances, no neighbours — just means, variances and a prior.

In [ ]:
gnb = GaussianNB().fit(Xtr_w, ytr_w)
print(f"GaussianNB on wine      : {gnb.score(Xte_w, yte_w):.3f}")
print(f"(K-NN scaled, for scale): {acc_scaled:.3f}")
print("\nNote: GaussianNB was given the RAW features and didn't care —")
print("it never computes a distance, so feature scales are irrelevant to it.")

### 4b · MultinomialNB on real SMS spam

5,572 real text messages, 13% of them spam. This is Naive Bayes' home turf: thousands of features (one per word), counts, and a task where K-NN's curse of dimensionality would be crippling.

In [ ]:
SMS_URL = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
sms = pd.read_csv(SMS_URL, sep="\t", header=None, names=["label", "message"])
sms["is_spam"] = (sms["label"] == "spam").astype(int)
print(sms.shape, "| spam rate:", round(sms["is_spam"].mean(), 3))
sms.head(3)

In [ ]:
Xtr_s, Xte_s, ytr_s, yte_s = train_test_split(
    sms["message"], sms["is_spam"], test_size=0.25, stratify=sms["is_spam"], random_state=RNG_SEED)

# CountVectorizer turns text into word counts — and it LEARNS the vocabulary,
# so it belongs inside the pipeline (Week 2's rule applies to text too).
spam_model = make_pipeline(CountVectorizer(), MultinomialNB())
spam_model.fit(Xtr_s, ytr_s)

y_pred_s = spam_model.predict(Xte_s)
print(f"accuracy: {accuracy_score(yte_s, y_pred_s):.3f}")
print("\nvocabulary size:", len(spam_model.named_steps['countvectorizer'].vocabulary_), "features")
print("\n", classification_report(yte_s, y_pred_s, target_names=["ham", "spam"]))

Look at the feature count: **thousands of dimensions**, trained in a fraction of a second, on a laptop. This is exactly the situation where K-NN's "nearest neighbour" would be meaningless — and Naive Bayes shrugs and gets on with it.

Read the classification report: accuracy alone would hide everything, because 87% of messages are ham. The per-class precision and recall for **spam** are the numbers that matter — and which one you'd rather protect is a value judgment (a real email lost in the junk folder vs a spam message getting through).

Let's see which words the model considers most spam-like:

In [ ]:
vec = spam_model.named_steps["countvectorizer"]
nb  = spam_model.named_steps["multinomialnb"]
log_ratio = nb.feature_log_prob_[1] - nb.feature_log_prob_[0]   # log P(word|spam) - log P(word|ham)
words = np.array(vec.get_feature_names_out())
top = np.argsort(log_ratio)[-15:][::-1]
print("Most spam-indicative words:")
print(", ".join(words[top]))

---
## Part 5 · ⏱ The stopwatch

The lecture's comparison table claimed: both train instantly, but K-NN *predicts* slowly while Naive Bayes predicts instantly. Let's verify that with a clock instead of trusting a slide.

In [ ]:
from sklearn.datasets import make_classification
Xb, yb = make_classification(n_samples=20000, n_features=30, n_informative=15,
                             n_classes=2, random_state=0)
Xb_tr, Xb_te = Xb[:16000], Xb[16000:]
yb_tr, yb_te = yb[:16000], yb[16000:]

results = []
for name, model in [("K-NN (k=5)", KNeighborsClassifier(5)), ("GaussianNB", GaussianNB())]:
    t0 = time.perf_counter(); model.fit(Xb_tr, yb_tr);  t_fit  = time.perf_counter() - t0
    t0 = time.perf_counter(); pred = model.predict(Xb_te); t_pred = time.perf_counter() - t0
    results.append({"model": name, "fit (s)": round(t_fit, 4), "predict 4k rows (s)": round(t_pred, 4),
                    "accuracy": round(accuracy_score(yb_te, pred), 3)})

pd.DataFrame(results)

**Read the table, then read it again.** Both fit almost instantly — but look at the prediction column. K-NN has to compare each of the 4,000 test rows against all 16,000 stored training rows; Naive Bayes does a handful of multiplications per row.

Expect roughly **two orders of magnitude** between them.

But now look at the accuracy column, because it complicates the story honestly: here K-NN is far *slower* **and** noticeably *more accurate*. Why? This synthetic data was generated with interacting informative features — exactly what Naive Bayes assumes away and what K-NN captures naturally. So the table isn't "one model wins"; it's a **trade-off you must price for your own problem**:

> Would you give up accuracy to predict a hundred times faster? On a research dataset — no. On a system inspecting 10,000 network packets per second — quite possibly yes, because a model that can't keep up isn't accurate, it's undeployed.

Accuracy against latency, memory and interpretability is a judgment you'll face in every real deployment, and there is no formula for it. That's why we practise it.

---
---
# 🏋️ Exercises (graded)

Replace each `...`, run every ✅ self-check, complete the disclosure.

### Exercise 1 — Distance by hand
For the two points below, compute with NumPy:
- **`d_euclid`** — Euclidean distance
- **`d_manhattan`** — Manhattan distance

*(No sklearn — use `np.sqrt`, `np.sum`, `np.abs`.)*

In [ ]:
a = np.array([2.0, 3.0, 6.0])
b = np.array([5.0, 7.0, 6.0])
d_euclid = ...
d_manhattan = ...
print(round(d_euclid, 3), round(d_manhattan, 3))

In [ ]:
# ✅ Self-check — Exercise 1
assert abs(d_euclid - 5.0) < 1e-6, "Euclidean: square the differences, sum, square root."
assert abs(d_manhattan - 7.0) < 1e-6, "Manhattan: sum of absolute differences."
print("Exercise 1 self-check passed ✔  (same 5-vs-7 pair as the lecture slide)")

### Exercise 2 — Your own K-NN, chosen honestly
Using the **Wine** training data (`Xtr_w`, `ytr_w`):
1. Build **`knn7`** — a pipeline of `StandardScaler` + `KNeighborsClassifier` with **K=7** and **`weights="distance"`**
2. Cross-validate it 5-fold on the training data → **`cv7`** (the array of scores)
3. Store the mean in **`cv7_mean`**

In [ ]:
knn7 = ...
cv7 = ...
cv7_mean = ...
print(cv7.round(3), '| mean:', round(cv7_mean, 3))

In [ ]:
# ✅ Self-check — Exercise 2
assert isinstance(knn7, Pipeline), "knn7 must be a Pipeline (scaler + KNN)."
p = knn7.get_params()
assert p["kneighborsclassifier__n_neighbors"] == 7, "Set n_neighbors=7."
assert p["kneighborsclassifier__weights"] == "distance", "Set weights='distance'."
assert len(cv7) == 5 and cv7_mean > 0.90, "5 folds expected; scaled K-NN on wine should score well above 0.90."
print(f"Exercise 2 self-check passed ✔  (CV mean {cv7_mean:.3f})")

### Exercise 3 — Quantify the scaling damage
Repeat the Part 3 experiment **as a proper CV comparison** on the Wine training data (not a single split):
- **`cv_raw`** — 5-fold CV scores for a bare `KNeighborsClassifier(5)` (no scaler)
- **`cv_scl`** — 5-fold CV scores for the same model inside a `StandardScaler` pipeline
- **`gap`** — the difference of the two means (`cv_scl.mean() - cv_raw.mean()`)

In [ ]:
cv_raw = ...
cv_scl = ...
gap = ...
print(f'unscaled {cv_raw.mean():.3f} ± {cv_raw.std():.3f}')
print(f'scaled   {cv_scl.mean():.3f} ± {cv_scl.std():.3f}')
print(f'gap      {gap:+.3f}')

In [ ]:
# ✅ Self-check — Exercise 3
assert len(cv_raw) == 5 and len(cv_scl) == 5, "Both need 5-fold CV on the TRAINING data."
assert gap > 0.10, "Scaling should improve wine K-NN by well over 10 percentage points."
print(f"Exercise 3 self-check passed ✔  — scaling is worth {gap*100:.1f} percentage points here.")

### Exercise 4 — Naive Bayes as a baseline
On the same Wine data, cross-validate **`GaussianNB()`** (5-fold, training data) into **`cv_nb`**, then set **`winner`** to `"knn"` or `"nb"` or `"tie"` based on this rule: if the two CV means differ by **less than one standard deviation** of the better model, call it `"tie"`; otherwise name the higher one.

In [ ]:
cv_nb = ...
diff = ...
better_std = ...
winner = ...
print(f'NB  {cv_nb.mean():.3f} ± {cv_nb.std():.3f}')
print(f'KNN {cv_scl.mean():.3f} ± {cv_scl.std():.3f}')
print('verdict:', winner)

In [ ]:
# ✅ Self-check — Exercise 4
assert len(cv_nb) == 5, "5-fold CV expected."
assert winner in ("knn", "nb", "tie"), 'winner must be "knn", "nb" or "tie".'
_diff = abs(cv_nb.mean() - cv_scl.mean())
_std = cv_nb.std() if cv_nb.mean() > cv_scl.mean() else cv_scl.std()
_expected = "tie" if _diff < _std else ("nb" if cv_nb.mean() > cv_scl.mean() else "knn")
assert winner == _expected, "Apply the stated rule to your own numbers — don't guess."
print(f"Exercise 4 self-check passed ✔  (verdict: {winner})")
print("Lesson: when two models are within a standard deviation, 'the better model' is a claim you cannot support.")

### Exercise 5 — Spam: which mistake would you rather make?
From the Part 4b spam model's predictions on the test set (`yte_s`, `y_pred_s`), compute for the **spam class**:
- **`spam_precision`** — of the messages flagged as spam, the fraction that really were spam
- **`spam_recall`** — of the real spam messages, the fraction that were caught

Compute them **from the confusion matrix** with NumPy (don't just read the report). Then set **`protect`** to `"precision"` or `"recall"`: for a *business* email account, which one would you protect?

In [ ]:
cm_s = confusion_matrix(yte_s, y_pred_s)   # rows: actual, cols: predicted
tn, fp, fn, tp = cm_s.ravel()
spam_precision = ...
spam_recall = ...
protect = '...'
print(cm_s)
print(f'precision {spam_precision:.3f} | recall {spam_recall:.3f}')

In [ ]:
# ✅ Self-check — Exercise 5
from sklearn.metrics import precision_score, recall_score
assert abs(spam_precision - precision_score(yte_s, y_pred_s)) < 1e-6, "Precision = TP / (TP + FP)."
assert abs(spam_recall - recall_score(yte_s, y_pred_s)) < 1e-6, "Recall = TP / (TP + FN)."
assert protect == "precision", "For a business inbox, a real email lost to the junk folder (FP) is the costly error."
print("Exercise 5 self-check passed ✔")

### Exercise 6 — Open-ended: tune, compare, and justify
Pick **one** investigation and carry it out properly (CV on training data only, mean ± spread reported):

**(a)** Tune `alpha` for the spam `MultinomialNB` across e.g. `[0.01, 0.1, 0.5, 1.0, 2.0]` — does Laplace smoothing matter here?
**(b)** Re-run the Part 2 K sweep on **Wine** instead of Iris — is the best K different, and is the curve flatter or sharper?
**(c)** Test the curse of dimensionality: add 200 columns of pure random noise to the Wine features and re-measure K-NN vs GaussianNB. Which degrades more?

Then write **3+ sentences** below: what you did, what you found, and one caveat about what your evidence does *not* establish.

In [ ]:
# your investigation here — (a), (b) or (c)
...

*Your write-up (≥ 3 sentences): what you did · what you found · one caveat.*

✍️ ...

---
## AI-usage disclosure (required)

> **AI tools used:** *e.g., "Claude — explained what weights='distance' does; helped debug a shape error in Exercise 6" — or — "None."*

## 📤 Submitting

1. **Runtime → Restart session and run all** — clean run, every self-check *passed*.
2. Disclosure filled, notebook renamed `Lab3_<student_id>`.
3. **File → Download → .ipynb** → upload to the MSTeam.

---
*Datasets: Iris and Wine (UCI, via scikit-learn) · SMS Spam Collection (Almeida & Hidalgo, UCI, free for research).*